### A5.4.8. Build Constant Folding Pass

**Problem:**

Build a graph transformation pass that identifies operations whose inputs are all constants, evaluates them at compile time, and replaces them with the precomputed constant result.

**Explanation:**

Constant folding eliminates work at runtime by computing results at compile time whenever all inputs are known constants. The pass walks the graph, finds operations with all-constant inputs, evaluates them, and replaces the operation node with a constant node holding the result.

How to build it:

1. Each node has a `value` field: constants have a value, operations start with `None`.
2. Walk the graph in topological order.
3. For each operation node, check if all input nodes have known values.
4. If yes, evaluate the operation using those values, set the node's value, and change its type to `"constant"`.
5. After the pass, dead operation nodes with no consumers can be removed.

**Example:**

Given `c = 3 + 4` followed by `z = c * x`, the pass computes `c = 7` at compile time, so at runtime only `z = 7 * x` remains.

In [ ]:
class Node:
    def __init__(self, name, op_type, inputs=None, value=None):
        self.name = name
        self.op_type = op_type
        self.inputs = inputs or []
        self.value = value

    def __repr__(self):
        if self.value is not None:
            return f"{self.name}(const={self.value})"
        input_names = [node.name for node in self.inputs]
        return f"{self.name}({self.op_type}, inputs={input_names})"


EVALUATORS = {
    "add": lambda values: values[0] + values[1],
    "mul": lambda values: values[0] * values[1],
    "neg": lambda values: -values[0],
}


def constant_fold(nodes):
    folded_count = 0
    for node in nodes:
        if node.op_type == "constant" or node.op_type == "input":
            continue
        all_inputs_constant = all(
            inp.value is not None
            for inp in node.inputs
        )
        if all_inputs_constant and node.op_type in EVALUATORS:
            input_values = [inp.value for inp in node.inputs]
            node.value = EVALUATORS[node.op_type](input_values)
            node.op_type = "constant"
            node.inputs = []
            folded_count += 1

    return folded_count


a = Node("a", "constant", value=3.0)
b = Node("b", "constant", value=4.0)
x = Node("x", "input")
c = Node("c", "add", [a, b])
d = Node("d", "neg", [b])
z = Node("z", "mul", [c, x])
w = Node("w", "add", [d, a])

all_nodes = [a, b, x, c, d, z, w]

print("Before constant folding:")
for node in all_nodes:
    print(f"  {node}")

folded = constant_fold(all_nodes)

print(f"\nAfter constant folding ({folded} nodes folded):")
for node in all_nodes:
    print(f"  {node}")

**References:**

📘 Aho, A. V. et al. (2006). *Compilers: Principles, Techniques, and Tools* — Constant Folding

---

[⬅️ Previous: Build Operator Fusion Pass](./07_build_operator_fusion_pass.ipynb) | [Next: Build Common Subexpression Elimination Pass ➡️](./09_build_common_subexpression_elimination_pass.ipynb)